**Problema de Negócio:** O SAC tem recebido muitas reclamações de pedidos atrasados. Os dados logísticos estão caóticos: motoristas preenchem o status de entrega com abreviações ("ent", "ENTREGUE", "Ent.") e os timestamps das previsões estão em UTC, enquanto o app do motorista registra em fuso local (-03:00).

**Objetivo:** Padronizar os status usando Expressões Regulares (Regex) no Python, alinhar as datas e usar SQL para calcular o tempo de atraso. O resultado final será identificar quais Estados (Regiões) possuem o maior índice (%) de quebra de SLA.

In [0]:
%python
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

estados = ['SP', 'RJ', 'MG', 'BA', 'RS', 'PR', 'SC', 'GO']
status_sujos = ['Entregue', 'entregue', 'ENT', 'ENTREGUE', 'Ent.', 'ent', 'Devolvido', 'Extraviado']

random.seed(42)
n_pedidos = 1000

df_pedidos = []
for i in range(n_pedidos):
    estado = random.choice(estados)
    
    dt_previsao = datetime(2024, 5, 1) + timedelta(days=random.randint(1, 30))
    
    if estado in ['BA', 'GO', 'RS']:
        atraso = timedelta(days=random.randint(0, 5)) 
    else:
        atraso = timedelta(days=random.randint(-2, 1)) 
        
    dt_entrega = dt_previsao + atraso
    
    df_pedidos.append({
        'order_id': f'ORD-{10000+i}',
        'estado_destino': estado,
        'data_previsao_utc': dt_previsao.strftime('%Y-%m-%d 10:00:00+00:00'), 
        'data_entrega_local': dt_entrega.strftime('%Y-%m-%d %H:%M:%S-03:00'), 
        'status_app': random.choice(status_sujos)
    })

spark_df = spark.createDataFrame(pd.DataFrame(df_pedidos))
spark_df.createOrReplaceTempView("bronze_pedidos")

display(spark_df)

order_id,estado_destino,data_previsao_utc,data_entrega_local,status_app
ORD-10000,RJ,2024-05-02 10:00:00+00:00,2024-05-02 00:00:00-03:00,ENTREGUE
ORD-10001,BA,2024-05-06 10:00:00+00:00,2024-05-11 00:00:00-03:00,entregue
ORD-10002,RJ,2024-05-20 10:00:00+00:00,2024-05-21 00:00:00-03:00,Entregue
ORD-10003,SP,2024-05-04 10:00:00+00:00,2024-05-03 00:00:00-03:00,ENTREGUE
ORD-10004,SP,2024-05-19 10:00:00+00:00,2024-05-18 00:00:00-03:00,Devolvido
ORD-10005,BA,2024-05-16 10:00:00+00:00,2024-05-20 00:00:00-03:00,Ent.
ORD-10006,SP,2024-05-26 10:00:00+00:00,2024-05-25 00:00:00-03:00,Devolvido
ORD-10007,PR,2024-05-10 10:00:00+00:00,2024-05-09 00:00:00-03:00,ENTREGUE
ORD-10008,PR,2024-05-05 10:00:00+00:00,2024-05-03 00:00:00-03:00,Devolvido
ORD-10009,RJ,2024-05-13 10:00:00+00:00,2024-05-13 00:00:00-03:00,Ent.


In [0]:
%python
from pyspark.sql import functions as F

df_bronze = spark.table("bronze_pedidos")


df_silver = df_bronze.withColumn("status_limpo", 
    F.when(F.upper(F.col("status_app")).rlike("^ENT"), "ENTREGUE")
     .otherwise(F.upper(F.col("status_app")))
)

df_silver = df_silver.filter(F.col("status_limpo") == "ENTREGUE")

df_silver = df_silver.withColumn("dt_previsao", F.to_timestamp(F.col("data_previsao_utc")).cast("date"))
df_silver = df_silver.withColumn("dt_entrega_real", F.to_timestamp(F.col("data_entrega_local")).cast("date"))

df_silver.createOrReplaceTempView("silver_pedidos")

display(df_silver.select("order_id", "status_app", "status_limpo", "dt_previsao", "dt_entrega_real"))

order_id,status_app,status_limpo,dt_previsao,dt_entrega_real
ORD-10000,ENTREGUE,ENTREGUE,2024-05-02,2024-05-02
ORD-10001,entregue,ENTREGUE,2024-05-06,2024-05-11
ORD-10002,Entregue,ENTREGUE,2024-05-20,2024-05-21
ORD-10003,ENTREGUE,ENTREGUE,2024-05-04,2024-05-03
ORD-10005,Ent.,ENTREGUE,2024-05-16,2024-05-20
ORD-10007,ENTREGUE,ENTREGUE,2024-05-10,2024-05-09
ORD-10009,Ent.,ENTREGUE,2024-05-13,2024-05-13
ORD-10010,entregue,ENTREGUE,2024-05-25,2024-05-26
ORD-10011,ent,ENTREGUE,2024-05-04,2024-05-04
ORD-10012,Entregue,ENTREGUE,2024-05-24,2024-05-24


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW gold_kpi_atrasos AS
SELECT 
    estado_destino,
    COUNT(order_id) AS total_pedidos,
    SUM(CASE WHEN datediff(dt_entrega_real, dt_previsao) > 0 THEN 1 ELSE 0 END) AS volume_atrasado,
    ROUND((SUM(CASE WHEN datediff(dt_entrega_real, dt_previsao) > 0 THEN 1 ELSE 0 END) / COUNT(order_id)) * 100, 1) AS percentual_atraso,
    ROUND(AVG(datediff(dt_entrega_real, dt_previsao)), 1) AS media_dias_variacao
FROM silver_pedidos
GROUP BY estado_destino;

SELECT * FROM gold_kpi_atrasos ORDER BY percentual_atraso DESC;

estado_destino,total_pedidos,volume_atrasado,percentual_atraso,media_dias_variacao
GO,98,86,87.8,2.7
BA,84,73,86.9,2.6
RS,100,80,80.0,2.4
PR,98,35,35.7,-0.3
SC,92,28,30.4,-0.4
MG,87,26,29.9,-0.4
RJ,100,27,27.0,-0.5
SP,93,22,23.7,-0.4


In [0]:
%sql
SELECT 
    estado_destino, 
    percentual_atraso,
    CASE 
        WHEN percentual_atraso > 40.0 THEN '🚨 Crítico (>40%)'
        ELSE '✅ No Prazo (<=40%)'
    END AS status_sla
FROM gold_kpi_atrasos 
ORDER BY percentual_atraso DESC;

estado_destino,percentual_atraso,status_sla
GO,87.8,🚨 Crítico (>40%)
BA,86.9,🚨 Crítico (>40%)
RS,80.0,🚨 Crítico (>40%)
PR,35.7,✅ No Prazo (<=40%)
SC,30.4,✅ No Prazo (<=40%)
MG,29.9,✅ No Prazo (<=40%)
RJ,27.0,✅ No Prazo (<=40%)
SP,23.7,✅ No Prazo (<=40%)


Databricks visualization. Run in Databricks to view.